<a href="https://colab.research.google.com/github/piaseckazaneta/Python/blob/main/Mapa_schronisk_dla_psow_BINGO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install folium pandas openpyxl plotly

In [2]:
from google.colab import files
uploaded = files.upload()

Saving schroniska_polska_geo.xlsx to schroniska_polska_geo.xlsx
Saving schroniska_statystyki_geo.xlsx to schroniska_statystyki_geo.xlsx


In [3]:
import pandas as pd
df_stats = pd.read_excel("schroniska_statystyki_geo.xlsx", sheet_name="statystyki")
df_adopsiaki = pd.read_excel("schroniska_statystyki_geo.xlsx", sheet_name="lokalizacje_adopsiaki")
df_firmy = pd.read_excel("schroniska_statystyki_geo.xlsx", sheet_name="lokalizacje_warsztatow_firmy")
df_schroniska = pd.read_excel("schroniska_polska_geo.xlsx", sheet_name="Schroniska w Polsce")

In [21]:
df_stats["rok"] = df_stats["rok"].astype(str)

In [4]:
df_stats.head()

,rok,zgloszenia_adopsiaki,rodziny_adopsiaki,dzieci_warsztaty,placowki_warsztaty,firmy_dfc,wolontariusze,linkedin,facebook,instagram,liczba_darczyncow,dane_czesciowe
0,2023,59,11,2726,44,5,7,1214,1242,1279,18,0
1,2024,77,45,2835,35,9,12,1834,1537,1630,22,0
2,2025,126,63,4846,62,14,16,2140,1976,1891,31,0
3,2026,30,3,385,5,2,0,0,0,0,0,1


In [5]:
df_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   rok                     4 non-null      int64
 1    zgloszenia_adopsiaki   4 non-null      int64
 2    rodziny_adopsiaki      4 non-null      int64
 3   dzieci_warsztaty        4 non-null      int64
 4    placowki_warsztaty     4 non-null      int64
 5    firmy_dfc              4 non-null      int64
 6   wolontariusze           4 non-null      int64
 7    linkedin               4 non-null      int64
 8    facebook               4 non-null      int64
 9    instagram              4 non-null      int64
 10   liczba_darczyncow      4 non-null      int64
 11  dane_czesciowe          4 non-null      int64
dtypes: int64(12)
memory usage: 516.0 bytes


In [6]:
df_adopsiaki.head()

,miasto,rok_od,X (EPSG:2180),Y (EPSG:2180),Lon (WGS84),Lat (WGS84)
0,Warszawa,2023,641394.81,487279.09,21.071149,52.233374
1,Łódź,2023,533004.32,434158.44,19.478486,51.772824
2,Brzeziny,2023,484051.23,397610.45,19.762611,51.800194
3,Wrocław,2023,358565.05,364114.54,16.978196,51.126311
4,Gdańsk,2025,477517.91,720521.53,18.654023,54.348291


In [7]:
df_firmy.head()

,miasto,rok,X (EPSG:2180),Y (EPSG:2180),Lon (WGS84),Lat (WGS84)
0,Łódź,2025,533004.32,434158.44,19.478486,51.772824
1,Warszawa,2025,641394.81,487279.09,21.071149,52.233374
2,Wrocław,2025,358565.05,364114.54,16.978196,51.126311
3,Opole,2025,424416.70,312967.60,17.929884,50.678793
4,Szczecin,2025,204574.42,627569.52,14.550962,53.430182


In [8]:
df_schroniska.head()

,Lp.,Nazwa schroniska,Miasto,Ulica / adres,Kod pocztowy,Miejscowość,Województwo,Telefon,E-mail,Strona WWW,X (EPSG:2180),Y (EPSG:2180),Lon (WGS84),Lat (WGS84)
0,1,Schronisko dla zwierząt Ciapkowo,Gdynia,ul. Małokacka 3A,81-654,Gdynia,pomorskie,58-622-25-52,schronisko@ciapkowo.pl,http://ciapkowo.pl/,474378.61,740009.05,18.604028,54.523330
1,2,Schronisko dla zwierząt w Tczewie,Tczew,ul. Malinowska,83-110,Tczew,pomorskie,058 531 96 56,tczew@otoz.pl,www.schroniskotczew.pl,485480.39,691856.65,18.777943,54.090876
2,3,OTOZ ANIMALS Schronisko w Dąbrówce k. Wejherowa,Luzino,Dąbrówka Młyn 30,84-242,Luzino,pomorskie,607-540-557,dabrowka@otoz.pl,www.schroniskodabrowka.pl,446531.71,745072.06,18.172778,54.566667
3,4,Schronisko dla zwierząt w Starogardzie Gd.,Starogard Gdański,ul. Hermanowska 24,83-200,Starogard Gdański,pomorskie,058 56 099 31,schroniskostarogard@otoz.pl,http://www.schroniskostarogard.pl,468907.10,679288.20,18.525774,53.977148
4,5,Schronisko dla zwierząt w Elblągu,Elbląg,ul. Królewiecka 233,82-300,Elbląg,warmińsko-mazurskie,055 2341646,elblag.schronisko@otoz.pl,www.schronisko.elblag.pl,528766.24,703939.10,19.441086,54.198900


BUDOWANIE MAPY

 1. Inicjalizacja mapy

In [9]:
import folium

mapa = folium.Map(
    location=[52.0, 19.5],
    zoom_start=6,
    tiles="OpenStreetMap"
)

2. Wyświetlenie schronisk na mapie Polski

In [13]:
import folium

# ZESZYT
mapa = folium.Map(location=[52.0, 19.5], zoom_start=6, tiles="CartoDB positron")

# KARTKA 1 – schroniska
warstwa_schroniska = folium.FeatureGroup(name="Schroniska")
for _, row in df_schroniska.iterrows():
    if row["Lat (WGS84)"] == "BRAK":
        continue
    folium.CircleMarker(
        location=[row["Lat (WGS84)"], row["Lon (WGS84)"]],
        radius=5,
        color="blue",
        fill=True,
        fill_opacity=0.6,
        popup=row["Nazwa schroniska"]
    ).add_to(warstwa_schroniska)
warstwa_schroniska.add_to(mapa)

# KARTKA 2 – adopsiaki
warstwa_adopsiaki = folium.FeatureGroup(name="#Adopsiaki")
for _, row in df_adopsiaki.iterrows():
    folium.CircleMarker(
        location=[row["Lat (WGS84)"], row["Lon (WGS84)"]],
        radius=5,
        color="orange",
        fill=True,
        fill_opacity=0.6,
        popup=row["miasto"]
    ).add_to(warstwa_adopsiaki)
warstwa_adopsiaki.add_to(mapa)

# KARTKA 3 – firmy
warstwa_firmy = folium.FeatureGroup(name="Warsztaty w firmach")
for _, row in df_firmy.iterrows():
    folium.CircleMarker(
        location=[row["Lat (WGS84)"], row["Lon (WGS84)"]],
        radius=5,
        color="green",
        fill=True,
        fill_opacity=0.6,
        popup=row["miasto"]
    ).add_to(warstwa_firmy)
warstwa_firmy.add_to(mapa)

# ZAKŁADKI – tylko raz, na samym końcu
folium.LayerControl().add_to(mapa)

mapa

In [14]:
mapa.save("mapa_fundacji.html")

In [15]:
from google.colab import files
files.download("mapa_fundacji.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Wykresy

In [17]:
import plotly.express as px

In [19]:
df_stats.columns = df_stats.columns.str.replace(" ", "")

In [22]:
fig = px.bar(
    df_stats,
    x="rok",
    y="zgloszenia_adopsiaki",
    title="Liczba zgłoszeń do projektu #Adopsiaki"
)
fig.show()

In [24]:
df_stats["etykieta"] = df_stats.apply(
    lambda row: str(row["zgloszenia_adopsiaki"]) + " *" if row["dane_czesciowe"] == 1
    else str(row["zgloszenia_adopsiaki"]),
    axis=1
)

fig = px.bar(
    df_stats,
    x="rok",
    y="zgloszenia_adopsiaki",
    text="etykieta",
    title="Liczba zgłoszeń do projektu #Adopsiaki"
)

fig.update_traces(
    textposition="outside",
    marker_color="#2C5F2E"
)

fig.add_annotation(
    text="* dane częściowe (I–II 2026)",
    xref="paper", yref="paper",
    x=1, y=-0.15,
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="right"
)

fig.show()

In [35]:
def zrob_wykres(df, kolumna, tytul):
    df = df.copy()
    df["etykieta"] = df.apply(
        lambda row: str(row[kolumna]) + " *" if row["dane_czesciowe"] == 1
        else str(row[kolumna]),
        axis=1
    )
    fig = px.bar(
        df,
        x="rok",
        y=kolumna,
        text="etykieta",
        title=tytul,
        color_discrete_sequence=["#2C5F2E"]
    )
    fig.update_traces(textposition="outside", marker_color="#2C5F2E")
    fig.add_annotation(
        text="* dane częściowe (I–II 2026)",
        xref="paper", yref="paper",
        x=1, y=-0.15,
        showarrow=False,
        font=dict(size=11, color="gray"),
        align="right"
    )
    return fig

fig1 = zrob_wykres(df_stats, "zgloszenia_adopsiaki", "Liczba zgłoszeń #Adopsiaki")
fig2 = zrob_wykres(df_stats, "rodziny_adopsiaki", "Liczba rodzin objętych wsparciem w projekcie #Adopsiaki")
fig3 = zrob_wykres(df_stats, "dzieci_warsztaty", "Liczba przeszkolonych dzieci")
fig4 = zrob_wykres(df_stats, "placowki_warsztaty", "Liczba szkół i przedszkoli objętych warsztatami")
fig5 = zrob_wykres(df_stats, "firmy_dfc", "Liczba firm, biorących udział w programie Dog Friendly Company")
fig6 = zrob_wykres(df_stats, "wolontariusze", "Liczba wolontariuszy w Fundacji")
fig7 = zrob_wykres(df_stats, "linkedin", "Liczba obserwujących na LinkedIn")
fig8 = zrob_wykres(df_stats, "facebook", "Liczba obserwujących na Facebooku")
fig9 = zrob_wykres(df_stats, "instagram", "Liczba obserwujących na Instagramie")
fig10 = zrob_wykres(df_stats, "liczba_darczyncow", "Liczba cyklicznych darczyńców")

fig1.show()
fig2.show()
fig3.show()
fig4.show()
fig5.show()
fig6.show()
fig7.show()
fig8.show()
fig9.show()
fig10.show()

In [36]:
w1 = fig1.to_html(full_html=False, include_plotlyjs=False)
w2 = fig2.to_html(full_html=False, include_plotlyjs=False)
w3 = fig3.to_html(full_html=False, include_plotlyjs=False)
w4 = fig4.to_html(full_html=False, include_plotlyjs=False)
w5 = fig5.to_html(full_html=False, include_plotlyjs=False)
w6 = fig6.to_html(full_html=False, include_plotlyjs=False)
w7 = fig7.to_html(full_html=False, include_plotlyjs=False)
w8 = fig8.to_html(full_html=False, include_plotlyjs=False)
w9 = fig9.to_html(full_html=False, include_plotlyjs=False)
w10 = fig10.to_html(full_html=False, include_plotlyjs=False)

mapa_html = mapa.get_root().render()

In [43]:
html_dashboard = f"""
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Working Dogs Foundation – Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: Arial, sans-serif; background: #f0f4f7; color: #333; }}

        .header {{
            background: #092e5a;
            color: white;
            padding: 20px 40px;
            display: flex;
            align-items: center;
            gap: 20px;
        }}
        .header h1 {{ font-size: 40px; font-weight: 500; }}
        .header p {{ font-size: 20px; opacity: 0.85; margin-top: 4px; }}

        .container {{ max-width: 1300px; margin: 0 auto; padding: 30px 20px; }}

        .section-title {{
            font-size: 18px;
            font-weight: 500;
            color: #2C5F2E;
            margin: 30px 0 15px 0;
            padding-bottom: 8px;
            border-bottom: 2px solid #2C5F2E;
        }}

        .map-wrapper {{
            border-radius: 10px;
            overflow: hidden;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
            height: 520px;
        }}
        .map-wrapper iframe {{
            width: 100%;
            height: 100%;
            border: none;
        }}

        .tabs {{
            display: flex;
            flex-wrap: wrap;
            gap: 8px;
            margin-bottom: 20px;
        }}
        .tab-btn {{
            padding: 8px 18px;
            border: 2px solid #2C5F2E;
            border-radius: 20px;
            background: white;
            color: #2C5F2E;
            cursor: pointer;
            font-size: 13px;
            font-family: Arial, sans-serif;
            transition: all 0.2s;
        }}
        .tab-btn:hover {{ background: #e8f5e9; }}
        .tab-btn.active {{ background: #2C5F2E; color: white; }}

        .tab-content {{ display: none; }}
        .tab-content.active {{ display: block; }}

        .chart-box {{
            background: white;
            border-radius: 10px;
            padding: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        }}

        .footer {{
            text-align: center;
            padding: 30px;
            color: #888;
            font-size: 12px;
            margin-top: 40px;
        }}
        .footer a {{ color: #2C5F2E; text-decoration: none; }}
    </style>
</head>
<body>

<div class="header">
    <div>
        <h1>Fundacja Working Dogs</h1>
        <p>Mapa schronisk i działań Fundacji w Polsce</p>
    </div>
</div>

<div class="container">

    <!-- MAPA -->
    <div class="section-title">Mapa</div>
    <div class="map-wrapper">
        {mapa_html}
    </div>

    <!-- WYKRESY -->
    <div class="section-title">Statystyki fundacji</div>

    <div class="tabs">
        <button class="tab-btn active" onclick="pokazTab('adopsiaki', this)">#Adopsiaki</button>
        <button class="tab-btn" onclick="pokazTab('warsztaty', this)">Warsztaty edukacyjne</button>
        <button class="tab-btn" onclick="pokazTab('dfc', this)">Dog Friendly Company</button>
        <button class="tab-btn" onclick="pokazTab('spolecznosc', this)">Społeczność</button>
    </div>

    <div id="adopsiaki" class="tab-content active">
        <div class="chart-box">{w1}</div>
        <div class="chart-box" style="margin-top:16px">{w2}</div>
    </div>

    <div id="warsztaty" class="tab-content">
        <div class="chart-box">{w3}</div>
        <div class="chart-box" style="margin-top:16px">{w4}</div>
    </div>

    <div id="dfc" class="tab-content">
        <div class="chart-box">{w5}</div>
    </div>

    <div id="spolecznosc" class="tab-content">
        <div class="chart-box">{w6}</div>
        <div class="chart-box" style="margin-top:16px">{w7}</div>
        <div class="chart-box" style="margin-top:16px">{w8}</div>
        <div class="chart-box" style="margin-top:16px">{w9}</div>
        <div class="chart-box" style="margin-top:16px">{w10}</div>
    </div>

</div>

<div class="footer">
    Dane: Working Dogs Foundation &nbsp;|&nbsp;
    <a href="https://workingdogs.pl/wsparcie" target="_blank">Wesprzyj fundację</a>
    &nbsp;|&nbsp; Mapa: © <a href="https://carto.com/" target="_blank">CartoDB</a>
</div>

<script>
function pokazTab(id, btn) {{
    document.querySelectorAll('.tab-content').forEach(t => t.classList.remove('active'));
    document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
    document.getElementById(id).classList.add('active');
    btn.classList.add('active');
}}
</script>

</body>
</html>
"""

with open("dashboard_workingdogs.html", "w", encoding="utf-8") as f:
    f.write(html_dashboard)

print("Zapisano dashboard_workingdogs.html")

Zapisano dashboard_workingdogs.html


In [44]:
from google.colab import files
files.download("dashboard_workingdogs.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>